In [1]:
# ================================
# Load Large Twitch Dataset (Custom Paths)
# ================================

import pandas as pd

# File paths (Windows paths with escaped backslashes)
edges_path = r"C:\Users\tuq24449\OneDrive - Temple University\Temple\2025\Spring '25\CIS 5524\Final Project\twitch_gamers\large_twitch_edges.csv"
features_path = r"C:\Users\tuq24449\OneDrive - Temple University\Temple\2025\Spring '25\CIS 5524\Final Project\twitch_gamers\large_twitch_features.csv"

# --- Load edges ---
try:
    edges = pd.read_csv(edges_path)
    print("✅ Edges loaded successfully.")
    print("📊 First 5 rows of edges (friendship links):")
    print(edges.head())
    print(f"🔗 Total number of edges: {len(edges)}\n")
except FileNotFoundError:
    print("❌ Could not find the edges file.")

# --- Load features ---
try:
    features_df = pd.read_csv(features_path, index_col=0)
    print("✅ Features loaded successfully.")
    print("📊 First 5 rows of features:")
    print(features_df.head())
    print(f"👤 Total number of users with features: {len(features_df)}\n")
except FileNotFoundError:
    print("❌ Could not find the features file.")

# --- Basic Info ---
print("🔍 Dataset Overview:")
print("Edge columns:", edges.columns.tolist())
print("Feature columns:", features_df.columns.tolist())


✅ Edges loaded successfully.
📊 First 5 rows of edges (friendship links):
   numeric_id_1  numeric_id_2
0         98343        141493
1         98343         58736
2         98343        140703
3         98343        151401
4         98343        157118
🔗 Total number of edges: 6797557

✅ Features loaded successfully.
📊 First 5 rows of features:
        mature  life_time  created_at  updated_at  numeric_id  dead_account  \
views                                                                         
7879         1        969  2016-02-16  2018-10-12           0             0   
500          0       2699  2011-05-19  2018-10-08           1             0   
382502       1       3149  2010-02-27  2018-10-12           2             0   
386          0       1344  2015-01-26  2018-10-01           3             0   
2486         0       1784  2013-11-22  2018-10-11           4             0   

       language  affiliate  
views                       
7879         EN          1  
500         

In [2]:
import networkx as nx

# Build an undirected graph from the edges
G = nx.from_pandas_edgelist(edges, source='numeric_id_1', target='numeric_id_2')

print(f"📈 Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")


📈 Graph created with 168114 nodes and 6797557 edges.


In [3]:
import random

# Sample positive edges from the graph
positive_pairs = random.sample(list(G.edges()), 10000)

# Create a fast lookup for all real edges
edge_set = set(map(tuple, map(sorted, G.edges())))

# Sample negative pairs (non-existent edges)
negative_pairs = set()
nodes = list(G.nodes())

while len(negative_pairs) < 10000:
    u, v = random.sample(nodes, 2)
    if (min(u, v), max(u, v)) not in edge_set:
        negative_pairs.add((u, v))

negative_pairs = list(negative_pairs)

print(f"✅ Sampled {len(positive_pairs)} positive and {len(negative_pairs)} negative pairs.")


✅ Sampled 10000 positive and 10000 negative pairs.


In [9]:
from networkx.algorithms.link_prediction import (
    adamic_adar_index, jaccard_coefficient, preferential_attachment
)

def compute_graph_features(G, u, v):
    # Common Neighbors
    cn = len(list(nx.common_neighbors(G, u, v)))

    # Adamic-Adar
    try:
        aa = list(adamic_adar_index(G, [(u, v)]))[0][2]
    except:
        aa = 0

    # Jaccard Similarity
    try:
        jc = list(jaccard_coefficient(G, [(u, v)]))[0][2]
    except:
        jc = 0

    # Preferential Attachment
    try:
        pa = list(preferential_attachment(G, [(u, v)]))[0][2]
    except:
        pa = 0

    return [cn, aa, jc, pa]


In [23]:
import random
from node2vec import Node2Vec

# Sample 3000 nodes from the full graph
sampled_nodes = random.sample(list(G.nodes()), 10000)
subG = G.subgraph(sampled_nodes).copy()

# Train Node2Vec on the subgraph
node2vec = Node2Vec(subG, dimensions=64, walk_length=10, num_walks=20, workers=2)
n2v_model = node2vec.fit(window=5, min_count=1)


Computing transition probabilities:   0%|          | 0/10000 [00:00<?, ?it/s]

In [25]:
# Make sure the subgraph has enough edges
assert len(subG.edges()) >= 10000, "Subgraph doesn't have enough real edges for 10,000 positive samples."

# ✅ 10,000 Positive pairs
positive_pairs = random.sample(list(subG.edges()), 10000)

# ✅ 10,000 Negative pairs
edge_set = set(map(tuple, map(sorted, subG.edges())))
nodes = list(subG.nodes())
negative_pairs = set()

while len(negative_pairs) < 10000:
    u, v = random.sample(nodes, 2)
    if (min(u, v), max(u, v)) not in edge_set:
        negative_pairs.add((u, v))

negative_pairs = list(negative_pairs)

print(f"✅ Sampled {len(positive_pairs)} positive and {len(negative_pairs)} negative pairs.")


✅ Sampled 10000 positive and 10000 negative pairs.


In [17]:
import numpy as np
import networkx as nx
from networkx.algorithms.link_prediction import adamic_adar_index

def hadamard_embedding(u, v, model):
    try:
        return model.wv[str(u)] * model.wv[str(v)]
    except KeyError:
        return None

def get_node_attributes(u, v, features_df):
    try:
        u_data = features_df[features_df['numeric_id'] == u].squeeze()
        v_data = features_df[features_df['numeric_id'] == v].squeeze()
        return [
            int(u_data['mature'] == v_data['mature']),
            int(u_data['affiliate'] == v_data['affiliate']),
            abs(u_data['life_time'] - v_data['life_time']),
        ]
    except:
        return [0, 0, 0]

def get_adamic_adar(u, v, G):
    try:
        score = list(adamic_adar_index(G, [(u, v)]))[0][2]
    except:
        score = 0
    return [score]


In [27]:
def build_all_variants(pairs, label):
    X_n2v, X_attr, X_aa = [], [], []
    y_all = []

    for u, v in pairs:
        emb = hadamard_embedding(u, v, n2v_model)
        if emb is None:
            continue

        X_n2v.append(emb)
        X_attr.append(np.concatenate([emb, get_node_attributes(u, v, features_df)]))
        X_aa.append(np.concatenate([emb, get_adamic_adar(u, v, subG)]))
        y_all.append(label)

    return X_n2v, X_attr, X_aa, y_all

X1_p, X2_p, X3_p, y_p = build_all_variants(positive_pairs, 1)
X1_n, X2_n, X3_n, y_n = build_all_variants(negative_pairs, 0)

X_n2v = X1_p + X1_n
X_attr = X2_p + X2_n
X_aa = X3_p + X3_n
y = y_p + y_n


In [31]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score
from xgboost import XGBClassifier
import numpy as np

def evaluate_model_avg(X, y, title, runs=20):
    acc_list, f1_list, auc_list = [], [], []

    for i in range(runs):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42 + i
        )
        clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', verbosity=0)
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]

        acc_list.append(accuracy_score(y_test, y_pred))
        f1_list.append(f1_score(y_test, y_pred))
        auc_list.append(roc_auc_score(y_test, y_proba))

    print(f"\n📌 {title} (averaged over {runs} runs):")
    print(f"✅ Accuracy:  {np.mean(acc_list):.4f}")
    print(f"✅ F1-score:  {np.mean(f1_list):.4f}")
    print(f"✅ AUC:       {np.mean(auc_list):.4f}")

# 🔍 Run evaluations for all three feature sets
evaluate_model_avg(X_n2v, y, "Node2Vec Only")
evaluate_model_avg(X_attr, y, "Node2Vec + Node Attributes")
evaluate_model_avg(X_aa, y, "Node2Vec + Adamic-Adar")



📌 Node2Vec Only (averaged over 20 runs):
✅ Accuracy:  0.9585
✅ F1-score:  0.9592
✅ AUC:       0.9920

📌 Node2Vec + Node Attributes (averaged over 20 runs):
✅ Accuracy:  0.9588
✅ F1-score:  0.9594
✅ AUC:       0.9919

📌 Node2Vec + Adamic-Adar (averaged over 20 runs):
✅ Accuracy:  0.9691
✅ F1-score:  0.9694
✅ AUC:       0.9944
